In [1]:
import glob
import pandas as pd

In [2]:
RESULT_DIR = 'aggregated/'
DATA_DIR = 'sentimental_flair/'

In [3]:
# https://www.geeksforgeeks.org/getting-all-csv-files-from-a-directory-using-python/
# csv files in the path 
files = glob.glob(DATA_DIR + "/*.csv") 
  
# defining an empty list to store  
# content 
df = pd.DataFrame() 
content = [] 
  
# checking all the csv files in the  
# specified path 
for filename in files: 
    
    # reading content of csv file 
    # content.append(filename) 
    df = pd.read_csv(filename, index_col=None) 
    content.append(df) 
  
# converting content to data frame 
df = pd.concat(content) 
df.reset_index(inplace=True)
print(df) 

    index                                           Headline  \
0       0  Travis Boyd packs production into latest oppor...   
1       1  Gordon Sondland, impeachment witness, accused ...   
2       2          Impeachment farce boomerangs on Democrats   
3       3  Robert Redford slams Donald Trump's 'dictator-...   
4       4  Pete Buttigieg camp to return donations from B...   
..    ...                                                ...   
74      7  88 Days of Recovery: How a Girls’ Soccer Team ...   
75      8  A College Student Was Killed by a Man Whose Ca...   
76      9                     Bernie Sanders vs. The Machine   
77     10  Trump Keeps Losing in Court. But His Legal Str...   
78     11  Russia Inquiry Review Is Expected to Undercut ...   

                  Source  Year  Month  Day  Pre-Covid  Bias  Cred  Sentiment  
0   The Washington Times  2019     11   28          0     1     2          1  
1   The Washington Times  2019     11   28          0     1     2        

In [4]:
result_df = pd.DataFrame(columns=['date_id','negative_l','total_l','negative_r','total_r'])
result_df = pd.DataFrame({
                        'date_id': pd.Series(dtype='str'),
                        'pre_covid': pd.Series(dtype='int'),
                        'negative_l': pd.Series(dtype='int'),
                        'total_l': pd.Series(dtype='int'),
                        'negative_r': pd.Series(dtype='int'),
                        'total_r': pd.Series(dtype='int'),
                        })
result_df

,date_id,pre_covid,negative_l,total_l,negative_r,total_r


In [5]:
for ind in df.index:
    date_id = str(df['Year'][ind]) + '-' + str(df['Month'][ind]) + '-' + str(df['Day'][ind])
    pre_covid = df['Pre-Covid'][ind]
    bias = df['Bias'][ind]
    sentiment = df['Sentiment'][ind]

    selected_df = result_df[result_df['date_id'] == date_id]
    if selected_df.empty: # new date for result_df
        row_data = []
        row_data.append(date_id) # date_id
        row_data.append(pre_covid) # pre_covid
        if bias == 0: # left
            if sentiment == -1:
                row_data.append(1) # negative_l : negative
                row_data.append(1) # total_l
                row_data.append(0) # negative_r
                row_data.append(0) # total_r
            else:
                row_data.append(0) # negative_l : neutral or positive
                row_data.append(1) # total_l
                row_data.append(0) # negative_r
                row_data.append(0) # total_r
        elif bias == 1: # right
            if sentiment == -1:
                row_data.append(0) # negative_l
                row_data.append(0) # total_l
                row_data.append(1) # negative_r : negative
                row_data.append(1) # total_r
            else:
                row_data.append(0) # negative_l
                row_data.append(0) # total_l
                row_data.append(0) # negative_r : neutral or positive
                row_data.append(1) # total_r
        row = pd.Series(row_data, index=result_df.columns)
        result_df = pd.concat([result_df, pd.DataFrame([row])], ignore_index=True)
    else: # date exists
        row_idx = result_df.index[result_df['date_id'] == date_id].tolist()[0]
        if bias == 0: # left
            if sentiment == -1:
                result_df.loc[row_idx, ['negative_l']] = result_df.loc[row_idx, ['negative_l']] + 1
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
            else:
                result_df.loc[row_idx, ['total_l']] = result_df.loc[row_idx, ['total_l']] + 1
        elif bias == 1: # right
            if sentiment == -1:
                result_df.loc[row_idx, ['negative_r']] = result_df.loc[row_idx, ['negative_r']] + 1
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
            else:
                result_df.loc[row_idx, ['total_r']] = result_df.loc[row_idx, ['total_r']] + 1
    

In [6]:
result_df

,date_id,pre_covid,negative_l,total_l,negative_r,total_r
0,2019-11-28,0,7,12,41,63
1,2019-11-12,0,4,4,0,0


In [ ]:
result_file = RESULT_DIR + 'aggregated.csv'
result_df.to_csv(result_file, index=False)